In [1]:
!pip install pymilvus tqdm requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.2/315.2 kB 21.9 MB/s eta 0:00:00


In [3]:
from google.colab import files
uploaded = files.upload()  # select your chunks.json here

import json
with open("chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Saving chunks.json to chunks (1).json
Loaded 160996 chunks


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

ZILLIZ_URI = os.getenv("ZILLIZ_URI")
ZILLIZ_TOKEN = os.getenv("ZILLIZ_TOKEN")
COLLECTION_NAME = os.getenv("COLLECTION_NAME")
EMBEDDING_URL = os.getenv("EMBEDDING_URL")
EMBEDDING_API_KEY = os.getenv("EMBEDDING_API_KEY")
EMBED_DIM         = 768
BATCH_SIZE        = 16

In [ ]:
from pymilvus import connections, Collection, CollectionSchema, FieldSchema, DataType

# connect to ZILLIZ DB
connections.connect(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)
print("Connected to Zilliz")

# defining schema
fields = [
    FieldSchema(name="id",          dtype=DataType.INT64,       is_primary=True, auto_id=True),
    FieldSchema(name="article_id",  dtype=DataType.INT64),
    FieldSchema(name="chunk_index", dtype=DataType.INT64),
    FieldSchema(name="text",        dtype=DataType.VARCHAR,      max_length=2000),
    FieldSchema(name="title",       dtype=DataType.VARCHAR,      max_length=500),
    FieldSchema(name="author",      dtype=DataType.VARCHAR,      max_length=200),
    FieldSchema(name="category",    dtype=DataType.VARCHAR,      max_length=200),
    FieldSchema(name="pub_date",    dtype=DataType.VARCHAR,      max_length=50),
    FieldSchema(name="embedding",   dtype=DataType.FLOAT_VECTOR, dim=EMBED_DIM),
]

# Create a table named altibbi_articles with this structure”
schema = CollectionSchema(fields, description="Altibbi Arabic medical articles")
col = Collection(name=COLLECTION_NAME, schema=schema)

# Create an index
col.create_index(
    field_name="embedding",
    index_params={"metric_type": "COSINE", "index_type": "AUTOINDEX"}
)

# Load the collection
col.load()
print("Collection created")

Connected to Zilliz
Dropped existing collection
Collection created


In [ ]:
import requests
import time
from tqdm.notebook import tqdm  # progress bar

def get_embeddings(texts, retries=3):
    """
    sends text to embedding API and returns vectors
    """
    headers = {"Authorization": EMBEDDING_API_KEY, "Content-Type": "application/json"}
    payload = {
        "docs": texts, # list of text chunks
        "dense_weight": 1.0, # semantic search
        "sparse_weight": 0.0, # traditional search (tf-idf)
        "convert_to_float32": True, # reduces memory usage
        "normalize_vectors": False, 
        "batch_size": BATCH_SIZE,
    }

    # retry 3 times if api fails
    for attempt in range(retries):
        try:
            resp = requests.post(EMBEDDING_URL, json=payload, headers=headers, timeout=60) # api call
            resp.raise_for_status()
            data = resp.json()
            return data["embeddings"]
        except Exception as e:
            wait = 2 ** attempt
            print(f"Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError("Embedding API failed.")

total = len(chunks)
for batch_start in tqdm(range(0, total, BATCH_SIZE), desc="Embedding"):
    batch = chunks[batch_start: batch_start + BATCH_SIZE]
    texts = [c["text"] for c in batch]

    try:
        embeddings = get_embeddings(texts)
    except RuntimeError as e:
        print(f"Failed at batch {batch_start}: {e}")
        break

    rows = []
    for i, (chunk, emb) in enumerate(zip(batch, embeddings)):
        m = chunk["metadata"]
        rows.append({
            "article_id":  int(m.get("article_id", 0)),
            "chunk_index": int(batch_start + i),
            "text":        chunk["text"][:2000],
            "title":       (m.get("title")    or "")[:500],
            "author":      (m.get("author")   or "")[:200],
            "category":    (m.get("category") or "")[:200],
            "pub_date":    (m.get("pub_date")  or "")[:50],
            "embedding":   emb,
        })

    col.insert(rows)

col.flush()
print(f"\nDone! {col.num_entities} vectors in Zilliz.")

Embedding:   0%|          | 0/10063 [00:00<?, ?it/s]


Done! 160996 vectors in Zilliz.
